In [2]:
from pathlib import Path
import pandas as pd

In [3]:
DATA_PATH = (
    Path("..")
    / "data"
    / "processed"
    / "daphnet_integrated.parquet"
)

print("Dataset found:", DATA_PATH.exists())

Dataset found: True


In [4]:
df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))

Dataset loaded successfully.
Rows: 1917887


In [5]:
preprocessed_df = df.drop(columns="source_file").copy()

print("Rows before preprocessing:", len(preprocessed_df))

Rows before preprocessing: 1917887


In [6]:
preprocessed_df = preprocessed_df[
    preprocessed_df["annotation"].isin([1, 2])
].copy()

print("Rows after removing outside-experiment samples:", len(preprocessed_df))

Rows after removing outside-experiment samples: 1140835


In [7]:
preprocessed_df["fog"] = preprocessed_df["annotation"].map({
    1: 0,
    2: 1
})

In [8]:
print("No FoG samples:", (preprocessed_df["fog"] == 0).sum())
print("FoG samples:", (preprocessed_df["fog"] == 1).sum())

No FoG samples: 1030050
FoG samples: 110785


In [9]:
preprocessed_df = preprocessed_df.sort_values(
    by=["participant_id", "recording_id", "time_ms"]
).reset_index(drop=True)

In [10]:
print(
    "Total missing values:",
    preprocessed_df.isna().sum().sum()
)

Total missing values: 0


Confirming TimeStamp Ordering

In [11]:
time_order_check = (
    preprocessed_df
    .groupby(["participant_id", "recording_id"])["time_ms"]
    .agg(lambda values: values.is_monotonic_increasing)
)

time_order_check

participant_id  recording_id
S01             R01             True
                R02             True
S02             R01             True
                R02             True
S03             R01             True
                R02             True
                R03             True
S04             R01             True
S05             R01             True
                R02             True
S06             R01             True
                R02             True
S07             R01             True
                R02             True
S08             R01             True
S09             R01             True
S10             R01             True
Name: time_ms, dtype: bool

In [12]:
print("Final rows:", len(preprocessed_df))
print("Final columns:", len(preprocessed_df.columns))
print(preprocessed_df.head())

Final rows: 1140835
Final columns: 14
   time_ms  ankle_forward_mg  ankle_vertical_mg  ankle_lateral_mg  \
0   750000               -30                990               326   
1   750015               -30               1000               356   
2   750031               -20                990               336   
3   750046               -20               1000               316   
4   750062                 0                990               316   

   thigh_forward_mg  thigh_vertical_mg  thigh_lateral_mg  trunk_forward_mg  \
0               -45                972               181               -38   
1               -18                981               212               -48   
2                18                981               222               -38   
3                36                990               222               -19   
4                36                990               212               -29   

   trunk_vertical_mg  trunk_lateral_mg  annotation participant_id  \
0        